In [ ]:
# -*- coding: utf-8 -*-
# ===============================
# GUI INFERENCE SCRIPT (M9 ADAPTIVE MULTIMODAL)
# ===============================
import os
import gc 
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True' 

import re, cv2, numpy as np, warnings
import tensorflow as tf
import librosa, whisper
from moviepy import VideoFileClip
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tkinter import Tk, filedialog

warnings.filterwarnings("ignore")

# ===============================
# CONFIG
# ===============================
MODEL_FILE = "best_model_m9_adaptive_cv.h5"
IMG_SIZE = (224, 224)
SPEC_SIZE = (128, 128)
MAX_TEXT_LEN = 100
SR = 22050
DURATION = 3

# ===============================
# ADAPTIVE FUSION LAYER (Wajib ada untuk Load Model M9)
# ===============================
@tf.keras.utils.register_keras_serializable()
class AdaptiveFusion(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.attention_net = models.Sequential([
            layers.Dense(32, activation='tanh'),
            layers.Dense(3, activation='softmax')
        ])
    def call(self, inputs):
        v, a, t = inputs
        combined = layers.concatenate([v, a, t])
        weights = self.attention_net(combined)
        wv, wa, wt = weights[:, 0:1], weights[:, 1:2], weights[:, 2:3]
        fused_output = layers.concatenate([v * wv, a * wa, t * wt])
        return [fused_output, weights]

# ===============================
# LOAD AI ENGINES & MODEL
# ===============================
print("Memuat mesin AI (Whisper STT & Model M9)...")
stt_model = whisper.load_model("base")

if not os.path.exists(MODEL_FILE):
    raise FileNotFoundError(
        f"\n[ERROR] File model '{MODEL_FILE}' tidak ditemukan!"
        "\nPastikan Anda sudah melatih dan menyimpan model M9 terlebih dahulu."
    )

# Load model lengkap dengan custom object AdaptiveFusion
base_model = tf.keras.models.load_model(MODEL_FILE, custom_objects={'AdaptiveFusion': AdaptiveFusion})

# Buat inference model untuk mengambil Output Prediksi dan Bobot Attention
fusion_layer_output = base_model.get_layer('fusion_layer').output[1] 
inference_model = models.Model(inputs=base_model.input, outputs=[base_model.output, fusion_layer_output])

# ===============================
# FEATURE EXTRACTION HELPERS
# ===============================
def extract_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()
    if not ret: return np.zeros((*IMG_SIZE, 3), dtype=np.float32)
    frame = cv2.resize(frame, IMG_SIZE)
    return (frame / 255.0).astype(np.float32)

def extract_audio_and_text(video_path):
    tmp = f"temp_proc_{os.getpid()}.wav"
    try:
        with VideoFileClip(video_path) as video:
            if video.audio is None: 
                video.close()
                return np.zeros((*SPEC_SIZE, 1), dtype=np.float32), ""
            video.audio.write_audiofile(tmp, logger=None)
            video.close() 
            
        y, sr = librosa.load(tmp, sr=SR)
        if len(y) < SR * DURATION: y = np.pad(y, (0, SR * DURATION - len(y)))
        y = y[:SR * DURATION]
        
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel = cv2.resize(librosa.power_to_db(mel), SPEC_SIZE)
        mel = np.expand_dims(mel, -1)
        mel = (mel - np.mean(mel)) / (np.std(mel) + 1e-9)
        
        text = ""
        try: text = stt_model.transcribe(tmp)['text']
        except: pass
        
        if os.path.exists(tmp): os.remove(tmp)
        return mel.astype(np.float32), text.lower()
    except Exception:
        if os.path.exists(tmp): os.remove(tmp)
        return np.zeros((*SPEC_SIZE, 1), dtype=np.float32), ""

def text_to_sequence(text):
    text = re.sub(r'[^a-zA-Z ]', '', text)
    seq = [ord(c) % 5000 for c in text][:MAX_TEXT_LEN]
    seq += [0] * (MAX_TEXT_LEN - len(seq))
    return np.array(seq)

# ===============================
# MAIN GUI & PREDICTION INTERFACE
# ===============================
def test_video_file():
    # Sembunyikan jendela utama tkinter yang kosong
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True) # Agar jendela dialog muncul di paling depan

    print("\n[INFO] Membuka jendela pemilihan file...")
    file_path = filedialog.askopenfilename(
        title="Pilih File Video untuk Diuji",
        filetypes=[("Video Files", "*.mp4 *.avi *.mkv *.mov"), ("All Files", "*.*")]
    )

    if not file_path:
        print("[INFO] Pemilihan file dibatalkan.")
        return

    filename = os.path.basename(file_path)
    print(f"\nMenganalisis video: {filename} ...")

    # Ekstraksi fitur dari video yang dipilih
    img = extract_frame(file_path)
    spec, txt_a = extract_audio_and_text(file_path)

    # Prediksi menggunakan model
    pred, weights = inference_model.predict([
        np.expand_dims(img, 0), 
        np.expand_dims(spec, 0),
        np.expand_dims(text_to_sequence(txt_a), 0)
    ], verbose=0)

    p = pred[0][0]
    w_v, w_a, w_t = weights[0][0], weights[0][1], weights[0][2]
    
    # Penentuan Status Klasifikasi
    status = 'Kasar' if p >= 0.5 else 'Non-Kasar'

    # Menampilkan Hasil Analisis di Console/Terminal
    print("\n" + "="*50)
    print("             HASIL ANALISIS VIDEO")
    print("="*50)
    print(f" Nama File      : {filename}")
    print(f" Status Klasifikasi : [{status}]")
    print(f" Tingkat Keyakinan  : {p*100:.2f}%")
    if txt_a.strip():
        print(f" Teks Transkripsi   : \"{txt_a}\"")
    else:
        print(f" Teks Transkripsi   : (Tidak ada suara/kata terdeteksi)")
    print("-" * 50)
    print(f" Kontribusi Bobot Adaptif:")
    print(f"   • Visual         : {w_v:.2f}")
    print(f"   • Audio          : {w_a:.2f}")
    print(f"   • Teks           : {w_t:.2f}")
    print("="*50 + "\n")

    # Bersihkan memori
    del img, spec, txt_a
    gc.collect()

if __name__ == "__main__":
    test_video_file()